In [ ]:
import contextlib
import copy
import csv
import json
import re
from collections import defaultdict
from difflib import SequenceMatcher
from functools import lru_cache
from math import log
from pathlib import Path

import cv2
import faiss
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from rapidfuzz.distance import Levenshtein
from rapidocr import LangRec, ModelType, OCRVersion, RapidOCR
from tqdm.auto import tqdm
from transformers import AutoModel, AutoProcessor
from ultralytics import YOLO

: 

In [ ]:
def find_root():
    p = Path().resolve()
    for cand in [p, *p.parents]:
        if (cand / "artifacts").exists():
            return cand
    raise FileNotFoundError("artifacts/ not found above " + str(p))


ROOT = find_root()


def _pick(name, *subs):
    for sub in subs:
        cand = ROOT / sub / name if sub else ROOT / name
        if cand.exists():
            return cand
    raise FileNotFoundError(name)


P = {
    "yolo_pt": _pick("best.pt", "artifacts\models"),
    "siglip_ft_pt": _pick("siglip_full_best.pt", "artifacts\models"),
    "recall_idx": _pick("gallery.faiss", "artifacts\index"),
    "recall_slugs": _pick("gallery_slugs.json", "artifacts"),
    "rerank_idx": _pick("gallery_finetuned.faiss", "artifacts\index"),
    "rerank_slugs": _pick("gallery_slugs_finetuned.json", "artifacts"),
    "catalog_csv": _pick("catalog_cleaned.csv", "data", ""),
    "gt_csv": _pick("eval_slugs_new.csv", "data", ""),
    "eval_dir": _pick("eval_with_slugs", "data", ""),
    "crops_dir": _pick("ref_crops", "data", ""),
}

CFG = {
    "model_name": "google/siglip2-base-patch16-384",
    "yolo_conf": 0.25,
    "siglip_k": 100,  # кандидатов, допускаемых до OCR-скоринга
    "top_k": 5,
    "rrf_k": 60,
    "w_base": 0.8,
    "w_ft": 0.2,
    "alpha": 0.6,
    "beta": 0.4,
    "ocr_norm": 40.0,
    "scoring": "linear",  # 'linear' | 'mul'
    "mode": "score",  # 'score' | 'rrf'
    "ocr_conf_min": 0.5,
    "fuzzy_ratio": 0.85,
    "box_w_cent": 0.45,
    "box_w_area": 0.35,
    "box_w_conf": 0.20,
    "box_edge_pen": 0.5,
    "box_margin": 0.02,
    "pen_cat": 0.5,
    "pen_color": 0.7,
    "seed": 42,
}

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
print("device:", device)

device: cuda


In [ ]:
base_processor = AutoProcessor.from_pretrained(CFG["model_name"])
base_model = AutoModel.from_pretrained(CFG["model_name"]).to(device).float().eval()

try:
    ft_state = torch.load(P["siglip_ft_pt"], map_location="cpu")
except FileNotFoundError as e:
    raise FileNotFoundError("нет дообученных весов: " + str(P["siglip_ft_pt"])) from e

ft_model = copy.deepcopy(base_model)
ft_model.load_state_dict(ft_state)
ft_model = ft_model.to(device).float().eval()

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

In [ ]:
idx_base = faiss.read_index(str(P["recall_idx"]))
with open(P["recall_slugs"], encoding="utf-8") as f:
    slugs_base = json.load(f)

idx_ft = faiss.read_index(str(P["rerank_idx"]))
with open(P["rerank_slugs"], encoding="utf-8") as f:
    slugs_ft = json.load(f)

In [ ]:
@torch.no_grad()
def embed_pair(img):
    x = base_processor(images=img.convert("RGB"), return_tensors="pt")
    pv = x["pixel_values"].to(device)
    ctx = (
        torch.autocast(device_type="cuda", dtype=torch.float16)
        if device == "cuda"
        else contextlib.nullcontext()
    )
    with ctx:
        out_b = base_model.get_image_features(pixel_values=pv, return_dict=True)
        out_f = ft_model.get_image_features(pixel_values=pv, return_dict=True)
    vb = out_b.pooler_output if hasattr(out_b, "pooler_output") else out_b
    vf = out_f.pooler_output if hasattr(out_f, "pooler_output") else out_f
    vb = F.normalize(vb.float(), dim=-1).cpu().numpy().flatten()
    vf = F.normalize(vf.float(), dim=-1).cpu().numpy().flatten()
    return vb, vf

In [10]:
def search_two(crop, k=None):
    k = k or CFG["siglip_k"]
    vb, vf = embed_pair(crop)
    D_b, I_b = idx_base.search(vb[None, :].astype("float32"), k)
    D_f, I_f = idx_ft.search(vf[None, :].astype("float32"), k)
    rank_b = [slugs_base[i] for i in I_b[0]]
    rank_f = [slugs_ft[i] for i in I_f[0]]
    cos_b = {slugs_base[i]: float(d) for d, i in zip(D_b[0], I_b[0])}
    cos_f = {slugs_ft[i]: float(d) for d, i in zip(D_f[0], I_f[0])}
    return rank_b, rank_f, cos_b, cos_f


def rrf(rankings, k=None):
    k = k or CFG["rrf_k"]
    s = {}
    for r in rankings:
        for pos, slug in enumerate(r, start=1):
            s[slug] = s.get(slug, 0.0) + 1.0 / (k + pos)
    return s


def score_fusion(cos_b, cos_f, w_base, w_ft):
    slugs = set(cos_b) | set(cos_f)
    return {s: w_base * cos_b.get(s, 0.0) + w_ft * cos_f.get(s, 0.0) for s in slugs}

In [ ]:
meta = pd.read_csv(P["catalog_csv"]).drop_duplicates("Slug").set_index("Slug")
NAME_BY_SLUG = meta["Название вина"].to_dict()
print("slugov:", len(meta))


def norm(s):
    s = str(s).lower().replace("ё", "е")
    s = re.sub(r"[^a-zа-я0-9]+", " ", s)
    return s.strip()


def is_year(token):
    return bool(re.fullmatch(r"(19|20)\d{2}", token))

slugov: 2103


In [ ]:
STOPWORDS = set(
    """вино вин год года урожай урожая россия россии произведено изготовлено
алк спирт об объём мл л литр литров алкоголь производство завод компания винодельня
виноград винограда сортовое столовое марочное выдержанное выдержка выдержку знак качество
гост ту декларация сертификат акциз штрих код номер дата розлив разлито бутылка бутылки
ёмкость температура хранение хранить употребить до после вскрытия охлаждать подавать
рекомендуется подача градусов градуса процентов процента оборотов оборота спирта
этиловый этилового ректификат ректификата дистиллят дистиллята виноградный виноградного
сусло сусла виноматериал виноматериала""".split()
)

FIELD_WEIGHTS = {
    "Название вина": 2.0,
    "Сорт винограда": 3.5,
    "Винодельня": 1.5,
    "Цвет": 2.0,
    "Категория": 1.5,
    "Регион": 3.0,
}

vocab = set()
token_to_slugs = defaultdict(set)
field_w = {}
slug_tokens = {}
for slug, row in meta.iterrows():
    toks = []
    for field, w in FIELD_WEIGHTS.items():
        for word in norm(row[field]).split():
            if len(word) < 3 or word.isdigit() or is_year(word) or word in STOPWORDS:
                continue
            toks.append((word, w, field))
            vocab.add(word)
            token_to_slugs[word].add(slug)
            field_w[word] = max(field_w.get(word, 0.0), w)
    slug_tokens[slug] = toks
N = len(meta)
idf = {w: log(1 + N / max(1, len(slugs))) for w, slugs in token_to_slugs.items()}
print("уникальных токенов:", len(vocab))
print(
    "токенов с несколькими slugami:",
    sum(1 for s in token_to_slugs.values() if len(s) > 1),
)

уникальных токенов: 1448
токенов с несколькими slugami: 946


In [ ]:
_CYR2LAT = dict(
    zip(
        "абвгдеёжзийклмнопрстуфхцчшщъыьэюя",
        [
            "a",
            "b",
            "v",
            "g",
            "d",
            "e",
            "e",
            "zh",
            "z",
            "i",
            "y",
            "k",
            "l",
            "m",
            "n",
            "o",
            "p",
            "r",
            "s",
            "t",
            "u",
            "f",
            "kh",
            "ts",
            "ch",
            "sh",
            "shch",
            "",
            "y",
            "",
            "e",
            "yu",
            "ya",
        ],
    )
)
_LAT2CYR2 = {
    "zh": "ж",
    "kh": "х",
    "ts": "ц",
    "ch": "ч",
    "sh": "ш",
    "shch": "щ",
    "yu": "ю",
    "ya": "я",
}
_LAT2CYR1 = {
    "a": "а",
    "b": "б",
    "c": "к",
    "d": "д",
    "e": "е",
    "f": "ф",
    "g": "г",
    "h": "х",
    "i": "и",
    "j": "й",
    "k": "к",
    "l": "л",
    "m": "м",
    "n": "н",
    "o": "о",
    "p": "п",
    "q": "к",
    "r": "р",
    "s": "с",
    "t": "т",
    "u": "у",
    "v": "в",
    "w": "в",
    "x": "кс",
    "y": "и",
    "z": "з",
}


def cyr_to_lat(s):
    return "".join(_CYR2LAT.get(c, c) for c in s)


def lat_to_cyr(s):
    out = []
    i = 0
    while i < len(s):
        if s[i : i + 2] in _LAT2CYR2:
            out.append(_LAT2CYR2[s[i : i + 2]])
            i += 2
        else:
            out.append(_LAT2CYR1.get(s[i], s[i]))
            i += 1
    return "".join(out)


def translit_variants(word):
    variants = {word}
    if all("а" <= c <= "я" or c == "ё" for c in word):
        variants.add(cyr_to_lat(word))
    elif all("a" <= c <= "z" for c in word):
        variants.add(lat_to_cyr(word))
    return variants


CANON = {
    "a": "a",
    "а": "a",
    "c": "c",
    "с": "c",
    "e": "e",
    "е": "e",
    "o": "o",
    "о": "o",
    "0": "o",
    "p": "p",
    "р": "p",
    "y": "y",
    "у": "y",
    "x": "x",
    "х": "x",
    "k": "k",
    "к": "k",
    "m": "m",
    "м": "m",
    "т": "m",
    "h": "h",
    "н": "h",
    "u": "u",
    "и": "u",
    "b": "b",
    "б": "b",
    "6": "b",
    "z": "z",
    "з": "z",
    "3": "z",
    "n": "n",
    "п": "n",
    "r": "r",
    "г": "r",
    "ь": "ь",
    "ъ": "ь",
}
_SKELETON_TABLE = str.maketrans(CANON)


def skeleton(w):
    return w.translate(_SKELETON_TABLE)


vocab_skeleton = defaultdict(set)
for w in vocab:
    vocab_skeleton[skeleton(w)].add(w)


@lru_cache(maxsize=None)
def canon(token, max_ed=1):
    forms = set()
    if token in vocab:
        forms.add(token)
    sk = skeleton(token)
    forms.update(vocab_skeleton.get(sk, ()))
    if len(sk) >= 4:
        for key, vals in vocab_skeleton.items():
            if (
                abs(len(key) - len(sk)) <= max_ed
                and Levenshtein.distance(sk, key) <= max_ed
            ):
                forms.update(vals)
    for v in translit_variants(token):
        if v in vocab:
            forms.add(v)
            forms.update(vocab_skeleton.get(skeleton(v), ()))
    return forms

In [ ]:
reader = RapidOCR(
    params={
        "Rec.lang_type": LangRec.CYRILLIC,
        "Rec.model_type": ModelType.MOBILE,
        "Rec.ocr_version": OCRVersion.PPOCRV5,
    }
)


def _ocr_unpack(res):
    if res is None:
        return [], []
    if isinstance(res, tuple) and len(res) == 3:
        _, txts, scores = res
        return list(txts or []), list(scores or [])
    if hasattr(res, "txts") and hasattr(res, "scores"):
        return list(res.txts or []), list(res.scores or [])
    raise TypeError("неизвестный формат результата RapidOCR: " + str(type(res)))


def preprocess_for_ocr(img):
    a = np.array(img)
    lab = cv2.cvtColor(a, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    out = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    mean = out.reshape(-1, 3).mean(axis=0, keepdims=True) + 1e-6
    return np.clip(out * (128.0 / mean), 0, 255).astype(np.uint8)  # white balance


def ocr_tokens(img, min_len=3):
    toks = []
    txts, scores = _ocr_unpack(reader(preprocess_for_ocr(img)))
    for text, conf in zip(txts, scores):
        conf = float(conf)
        if conf < CFG["ocr_conf_min"]:
            continue
        for w in norm(text).split():
            if len(w) < min_len or w.isdigit() or is_year(w) or w in STOPWORDS:
                continue
            toks.append((w, conf))
    return toks

[INFO] 2026-09-25 10:49:21,538 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-25 10:49:21,775 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Sergey\Documents\VS Code Projects\lct-2026\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-09-25 10:49:21,777 [RapidOCR] main.py:63: Using C:\Users\Sergey\Documents\VS Code Projects\lct-2026\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-09-25 10:49:21,899 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-25 10:49:21,926 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Sergey\Documents\VS Code Projects\lct-2026\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-25 10:49:21,927 [RapidOCR] main.py:63: Using C:\Users\Sergey\Documents\VS Code Projects\lct-2026\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-25 10:49:22,009 [RapidOCR] 

In [ ]:
COLOR_ALIASES = {
    "белый": "белое",
    "белая": "белое",
    "белое": "белое",
    "беаый": "белое",
    "6eлый": "белое",
    "красный": "красное",
    "красное": "красное",
    "розовый": "розовое",
    "розовое": "розовое",
    "оранжевый": "оранжевое",
    "оранжевое": "оранжевое",
}


def _s(v):
    return "" if pd.isna(v) else str(v).lower()


def color_penalty(toks, row):
    det = None
    for w, _ in toks:
        if w in COLOR_ALIASES:
            det = COLOR_ALIASES[w]
            break
    if det is None:
        return 1.0
    pen = 1.0
    cat, col = _s(row.get("Категория")), _s(row.get("Цвет"))
    if cat and det not in cat:
        pen *= CFG["pen_cat"]
    if col and det not in col:
        pen *= CFG["pen_color"]
    return pen

In [ ]:
def query_forms(ocr_toks):
    return [(ot, oc, canon(ot), translit_variants(ot)) for ot, oc in ocr_toks]


def fuzzy_match_variants(a_vars, b_vars):
    best = 0.0
    for a in a_vars:
        for b in b_vars:
            best = max(best, SequenceMatcher(None, a, b).ratio())
    if best >= CFG["fuzzy_ratio"]:
        return True, best
    return False, 0.0


def ocr_score(slug, forms):
    score = 0.0
    for ot, oc, cf, ot_variants in forms:
        if cf:
            per_slug = 0.0
            for ftok in cf:
                if ftok in token_to_slugs and slug in token_to_slugs[ftok]:
                    per_slug = max(
                        per_slug, idf.get(ftok, 0.0) * field_w.get(ftok, 1.0)
                    )
            if per_slug > 0:
                score += per_slug * oc
                continue
        best = 0.0
        for ct, cw, field in slug_tokens[slug]:
            matched, ratio = fuzzy_match_variants(ot_variants, translit_variants(ct))
            if matched:
                best = max(best, cw * idf.get(ct, 1.0) * ratio)
        if best > 0:
            score += best * oc
    return score

In [17]:
def pick_main_box(boxes, img_w, img_h):
    if not boxes:
        return None
    mx, my = img_w * CFG["box_margin"], img_h * CFG["box_margin"]
    scored = []
    for b in boxes:
        x0, y0, x1, y1 = b["box"]
        cx = (x0 + x1) / 2 / img_w - 0.5
        cy = (y0 + y1) / 2 / img_h - 0.5
        cent = 1 - (cx * cx + cy * cy) ** 0.5
        area = (x1 - x0) * (y1 - y0) / (img_w * img_h)
        s = (
            cent * CFG["box_w_cent"]
            + area * CFG["box_w_area"]
            + b["conf"] * CFG["box_w_conf"]
        )
        if x0 < mx or y0 < my or x1 > img_w - mx or y1 > img_h - my:
            s *= CFG["box_edge_pen"]
        scored.append((s, b))
    scored.sort(key=lambda x: -x[0])
    return scored[0][1]

In [18]:
yolo = YOLO(str(P["yolo_pt"]))


def full_search_features(qimg):
    r = yolo(qimg, conf=CFG["yolo_conf"], verbose=False)[0]
    if len(r.boxes) == 0:
        return None
    boxes = [
        {"conf": float(b.conf[0]), "box": tuple(b.xyxy[0].cpu().numpy().astype(int))}
        for b in r.boxes
    ]
    best = pick_main_box(boxes, qimg.width, qimg.height)
    crop = qimg.crop(best["box"])
    rank_b, rank_f, cos_b, cos_f = search_two(crop)
    ocr_toks = ocr_tokens(crop)
    return {
        "crop": crop,
        "rank_b": rank_b,
        "rank_f": rank_f,
        "cos_b": cos_b,
        "cos_f": cos_f,
        "ocr_toks": ocr_toks,
        "yolo_conf": best["conf"],
    }

In [ ]:
def full_search_from_features(feat, cfg=None, **over):
    cfg = {**(cfg or CFG), **over}
    if cfg["mode"] == "rrf":
        fused = rrf([feat["rank_b"], feat["rank_f"]], cfg["rrf_k"])
    else:
        fused = score_fusion(feat["cos_b"], feat["cos_f"], cfg["w_base"], cfg["w_ft"])
    mx = max(fused.values()) if fused else 1.0
    sig_scores = {s: v / mx for s, v in fused.items()}
    cand = sorted(sig_scores.items(), key=lambda x: -x[1])[: cfg["siglip_k"]]
    forms = query_forms(feat["ocr_toks"])
    scored = []
    for slug, sig_score in cand:
        o_score = ocr_score(slug, forms)
        if slug in meta.index:
            o_score *= color_penalty(feat["ocr_toks"], meta.loc[slug])
        ocr_n = o_score / (o_score + cfg["ocr_norm"])
        if cfg["scoring"] == "linear":
            final = cfg["alpha"] * sig_score + cfg["beta"] * ocr_n
        else:
            final = sig_score * (0.5 + 0.5 * ocr_n)
        scored.append(
            {"slug": slug, "siglip": sig_score, "ocr": o_score, "final": final}
        )
    scored.sort(key=lambda x: -x["final"])
    return {
        "crop": feat["crop"],
        "ocr_toks": [t for t, _ in feat["ocr_toks"]],
        "top": scored[: cfg["top_k"]],
        "yolo_conf": feat["yolo_conf"],
    }

In [ ]:
gt = pd.read_csv(P["gt_csv"])
gt = gt[gt["слаг"] != "нет в каталоге"].copy()
gt["stem"] = gt["имя_файла"].astype(str).str.strip()
gt_map = dict(zip(gt["stem"], gt["слаг"]))
print("ground truth:", len(gt_map))

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}
paths = sorted(p for p in P["eval_dir"].glob("*") if p.suffix.lower() in IMG_EXT)
print("фото:", len(paths))

features = {}
for p in tqdm(paths, desc="features"):
    try:
        img = Image.open(p).convert("RGB")
        f = full_search_features(img)
        if f is not None:
            features[p.stem] = f
    except Exception as e:
        print(p.name, "->", type(e).__name__, e)
print("features:", len(features), "/", len(paths))

ground truth: 55
фото: 61


features:   0%|          | 0/61 [00:00<?, ?it/s]

features: 61 / 61


In [ ]:
def evaluate(gt_map, features, **over):
    cfg = {**CFG, **over}
    hits1 = hits_topk = n = 0
    per_img = []
    for stem, true in gt_map.items():
        f = features.get(stem)
        if f is None:
            continue
        res = full_search_from_features(f, cfg)
        preds = [it["slug"] for it in res["top"]]
        per_img.append((stem, true, preds))
        n += 1
        if preds[:1] == [true]:
            hits1 += 1
        if true in preds:
            hits_topk += 1
    return hits1 / max(n, 1), hits_topk / max(n, 1), per_img


def bootstrap_ci(per_img, n=2000, seed=42):
    hits = np.array([p[:1] == [t] for _, t, p in per_img], dtype=float)
    rng = np.random.default_rng(seed)
    means = hits[rng.integers(0, len(hits), (n, len(hits)))].mean(1)
    return np.percentile(means, [2.5, 97.5])


acc1_lin, acck_lin, per_lin = evaluate(gt_map, features, scoring="linear")
acc1_mul, acck_mul, per_mul = evaluate(gt_map, features, scoring="mul")
tk = CFG["top_k"]
print("linear: top1=%.4f top%d=%.4f" % (acc1_lin, tk, acck_lin))
print("mul:    top1=%.4f top%d=%.4f" % (acc1_mul, tk, acck_mul))
lo, hi = bootstrap_ci(per_lin)
print("linear top1 CI95: [%.3f, %.3f]" % (lo, hi))

print("=== alpha sweep (ocr_norm=40) ===")
for a in [0.4, 0.5, 0.6, 0.7]:
    acc1, acck, _ = evaluate(
        gt_map, features, scoring="linear", alpha=a, beta=1 - a, ocr_norm=40
    )
    print("alpha=%.1f: top1=%.4f top10=%.4f" % (a, acc1, acck))

linear: top1=0.9091 top10=0.9818
mul:    top1=0.9091 top10=0.9818
linear top1 CI95: [0.818, 0.982]
=== alpha sweep (ocr_norm=40) ===
alpha=0.4: top1=0.8909 top10=0.9818
alpha=0.5: top1=0.9091 top10=0.9818
alpha=0.6: top1=0.9091 top10=0.9818
alpha=0.7: top1=0.8545 top10=0.9818


In [22]:
def agreement(k=10, n_max=20):
    same_top1 = same_half = n = 0
    for stem in list(features)[:n_max]:
        rb, rf = features[stem]["rank_b"], features[stem]["rank_f"]
        if not rb or not rf:
            continue
        n += 1
        if rb[0] == rf[0]:
            same_top1 += 1
        if len(set(rb[:k]) & set(rf[:k])) >= k // 2:
            same_half += 1
    nn = max(n, 1)
    print(
        "n=%d top1-agreement=%.2f top-%d-overlap>=50%%=%.2f"
        % (n, same_top1 / nn, k, same_half / nn)
    )


agreement()

n=20 top1-agreement=0.60 top-10-overlap>=50%=0.30


In [ ]:
OUTPUT_DIR = ROOT / "eval_vis_ocr_v4"
OUTPUT_DIR.mkdir(exist_ok=True)

log = []
for p in tqdm(paths, desc="render"):
    f = features.get(p.stem)
    if f is None:
        log.append({"file": p.name, "status": "no_box"})
        continue
    res = full_search_from_features(f)
    vis = cv2.cvtColor(np.array(res["crop"]), cv2.COLOR_RGB2BGR)
    H, W = vis.shape[:2]
    pad = 24 * (3 + len(res["top"])) + 10
    canvas = np.full((H + pad, W, 3), 255, dtype=np.uint8)
    canvas[:H] = vis
    yc = res["yolo_conf"]
    lines = [
        p.name + " (yolo=%.2f)" % yc,
        "OCR: " + ", ".join(res["ocr_toks"][:8]),
        "-",
    ]
    for rank, item in enumerate(res["top"], 1):
        name = NAME_BY_SLUG.get(item["slug"], "?")
        fin = item["final"]
        lines.append("%d. %s | %.3f" % (rank, name, fin))
    y = H + 22
    for ln in lines:
        cv2.putText(
            canvas,
            ln,
            (6, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 0, 0),
            1,
            cv2.LINE_AA,
        )
        y += 22
    Image.fromarray(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)).save(
        OUTPUT_DIR / (p.stem + ".jpg"), quality=85
    )
    row = {"file": p.name, "status": "ok", "yolo_conf": round(res["yolo_conf"], 3)}
    row["ocr_toks"] = "|".join(res["ocr_toks"][:10])
    for rank, item in enumerate(res["top"], 1):
        row["top%d_slug" % rank] = item["slug"]
        row["top%d_final" % rank] = round(item["final"], 4)
    log.append(row)

all_keys = set()
for r in log:
    all_keys.update(r)
with open(ROOT / "eval_ocr_results_v4.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=sorted(all_keys))
    w.writeheader()
    w.writerows(log)
ok = sum(1 for r in log if r.get("status") == "ok")
print("готово. всего:", len(log), "обработано:", ok)
print("визуализация ->", OUTPUT_DIR)

render:   0%|          | 0/61 [00:00<?, ?it/s]

готово. всего: 61 обработано: 61
визуализация -> C:\Users\Sergey\Documents\VS Code Projects\lct-2026\eval_vis_ocr_v4


In [ ]:
pred = pd.read_csv(ROOT / "eval_ocr_results_v4.csv")
pred["stem"] = pred["file"].astype(str).str.rsplit(".", n=1).str[0]
gt2 = pd.read_csv(P["gt_csv"])
gt2 = gt2[gt2["слаг"] != "нет в каталоге"].copy()
gt2["stem"] = gt2["имя_файла"].astype(str).str.strip()
gt2 = gt2[["stem", "слаг"]].rename(columns={"слаг": "true_slug"})
cols = ["stem"] + ["top%d_slug" % i for i in range(1, 11)]
cols = [c for c in cols if c in pred.columns]
merged = gt2.merge(pred[cols], on="stem", how="inner")


def rank_of_true(row):
    for rank in range(1, 11):
        if row.get("top%d_slug" % rank) == row["true_slug"]:
            return rank
    return 11


merged["rank"] = merged.apply(rank_of_true, axis=1)
n = len(merged)
print("всего:", n)
for r in range(1, 11):
    cnt = (merged["rank"] == r).sum()
    print("top-%d: %d (%.1f%%)" % (r, cnt, 100 * cnt / n))
miss = (merged["rank"] == 11).sum()
print("не в top-10: %d (%.1f%%)" % (miss, 100 * miss / n))
r1 = (merged["rank"] == 1).sum()
r10 = (merged["rank"] <= 10).sum()
print("F1@1 = %.4f" % (r1 / n))
print("top-10 accuracy = %.4f" % (r10 / n))

recall_miss = sum(
    1
    for stem, true in gt_map.items()
    if stem in features
    and true not in set(features[stem]["cos_b"]) | set(features[stem]["cos_f"])
)
print("recall-miss (цель вне визуального top-100):", recall_miss)

errors = merged[merged["rank"] > 1][
    ["stem", "true_slug", "top1_slug", "rank"]
].sort_values("rank")
errors.to_csv(ROOT / "eval_errors_v4.csv", index=False)
print("ошибок:", len(errors))

всего: 55
top-1: 50 (90.9%)
top-2: 3 (5.5%)
top-3: 1 (1.8%)
top-4: 0 (0.0%)
top-5: 0 (0.0%)
top-6: 0 (0.0%)
top-7: 0 (0.0%)
top-8: 0 (0.0%)
top-9: 0 (0.0%)
top-10: 0 (0.0%)
не в top-10: 1 (1.8%)
F1@1 = 0.9091
top-10 accuracy = 0.9818
recall-miss (цель вне визуального top-100): 0
ошибок: 5
